# 03 · Augmentation & generation
Build the augmented prompt from retrieved chunks (numbered context), generate with the **heavy** model, then verify grounding. This is the manual version of what the LangGraph agent automates in notebook 04.

In [ ]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
from dotenv import load_dotenv
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# Automatically load environment variables from rag_service/.env and repo-root .env
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT.parent / ".env", override=False)

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env (default when keys are present)
has_keys = bool(os.getenv("GEMINI_API_KEY") or os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY"))
OFFLINE = (os.getenv("NB_OFFLINE", "0" if has_keys else "1") == "1")
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings, reset_settings_cache
reset_settings_cache()
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

In [ ]:
from app.retriever import HybridRetriever, build_where
from app.agent import _ctx_block, NO_ANSWER
from app.providers import get_llm, llm_text
question = "Which organelle produces ATP and what kind of DNA does it have?"
hits = HybridRetriever(where=build_where(["BIO101"]), top_k=3).search(question)
relevant = [{"doc": d, "score": s} for d, s in hits]
print(_ctx_block(relevant)[:600], "...")

## Augmented prompt

In [ ]:
prompt = ("TASK:ANSWER\nYou are a university course tutor. Answer ONLY from the numbered context. Cite like [1]. "
          "If the context lacks the answer reply exactly INSUFFICIENT_CONTEXT.\n"
          f"CONTEXT:\n{_ctx_block(relevant)}\nQUESTION: {question}\n")
print(len(prompt), "chars sent to the LLM")

## Generation (heavy tier)

In [ ]:
answer = llm_text(get_llm("heavy").invoke(prompt)).strip()
print(answer)

## Grounding verification (is every claim supported by the context?)

In [ ]:
from app.agent import _json
v = _json(llm_text(get_llm("heavy").invoke(
    "TASK:VERIFY\nReply ONLY JSON {\"supported_ratio\":0-1,\"verdict\":\"grounded|ungrounded\"}.\n"
    f"ANSWER: {answer}\nCONTEXT: {_ctx_block(relevant)}")))
print(v)

## Fail-closed demo: a question the material cannot answer

In [ ]:
q2 = "Who won the football world cup in 1998?"
h2 = HybridRetriever(where=build_where(["BIO101"]), top_k=3).search(q2)
p2 = prompt.split("CONTEXT:")[0] + f"CONTEXT:\n{_ctx_block([{'doc':d,'score':s} for d,s in h2])}\nQUESTION: {q2}\n"
print(llm_text(get_llm("heavy").invoke(p2)).strip())